# Contexto y memoria del cliente


## 1. Instalar dependencias


In [ ]:
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("Selecciona un runtime con Python 3.11 o superior.")

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "pydantic-ai-slim[openrouter]==2.68.0", "nest_asyncio",
])


## 2. Verificar versiones y preparar Colab


In [ ]:
from importlib.metadata import version
import platform
import nest_asyncio

print("Python:", platform.python_version())
print("pydantic-ai-slim:", version("pydantic-ai-slim"))
print("pydantic:", version("pydantic"))
print("nest-asyncio:", version("nest-asyncio"))
nest_asyncio.apply()


## 3. Definir los clientes


In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from pydantic_ai import Agent, RunContext

class Cliente(BaseModel):
    model_config = ConfigDict(frozen=True)
    cliente_id: str = Field(min_length=1)
    nombre: str = Field(min_length=1)
    presupuesto_mxn: int = Field(ge=0, strict=True)
    estilo: Literal["breve", "detallado"] = "breve"

ana = Cliente(cliente_id="ana", nombre="Ana", presupuesto_mxn=100)
bruno = Cliente(
    cliente_id="bruno", nombre="Bruno", presupuesto_mxn=250,
    estilo="detallado",
)
print(ana.model_dump())


## 4. Crear el agente, el contexto y la herramienta


In [ ]:
agente = Agent(
    deps_type=Cliente,
    system_prompt=(
        "Eres Don Plátano, corredor teatral del Banano Exchange. "
        "Responde en español con humor ligero. Este es un ejercicio ficticio. "
        "No inventes datos del cliente ni compras anteriores. "
        "Para consultar el presupuesto usa resumen_presupuesto. "
        "No ejecutas compras ni modificas saldos. "
        "Si una referencia no aparece en el historial, pide aclaración."
    ),
)

@agente.system_prompt(dynamic=True)
def instrucciones_cliente(ctx: RunContext[Cliente]) -> str:
    cliente = ctx.deps
    return (
        f"Cliente actual: {cliente.nombre}. "
        f"Estilo de respuesta: {cliente.estilo}. "
        "El perfil no contiene el objeto de interés de la conversación."
    )

@agente.tool
def resumen_presupuesto(ctx: RunContext[Cliente]) -> dict:
    """Consulta el presupuesto ficticio y su equivalencia en bananas a 5 MXN cada una."""
    return {
        "presupuesto_mxn": ctx.deps.presupuesto_mxn,
        "precio_ficticio_banana_mxn": 5,
        "bananas_enteras": ctx.deps.presupuesto_mxn // 5,
    }


## 5. Probar sin llamadas al modelo


In [ ]:
from pydantic_ai import models
from pydantic_ai.models.test import TestModel
from pydantic_ai.messages import SystemPromptPart, ToolReturnPart, UserPromptPart

# Recuperamos las partes de cada mensaje, sin depender del texto final simulado.
def partes(resultado):
    return [parte for mensaje in resultado.all_messages() for parte in mensaje.parts]

permitidas_antes = models.ALLOW_MODEL_REQUESTS
models.ALLOW_MODEL_REQUESTS = False
try:
    with agente.override(model=TestModel()):
        prueba_ana = agente.run_sync("Consulta mi presupuesto.", deps=ana)
        prueba_bruno = agente.run_sync("Consulta mi presupuesto.", deps=bruno)

    for resultado, esperado in [(prueba_ana, 20), (prueba_bruno, 50)]:
        devoluciones = [
            p.content for p in partes(resultado)
            if isinstance(p, ToolReturnPart) and p.tool_name == "resumen_presupuesto"
        ]
        assert devoluciones, "No se registró la devolución de la herramienta."
        assert devoluciones[-1]["bananas_enteras"] == esperado

    # El simulador devuelve texto fijo: no está recordando este objeto.
    with agente.override(model=TestModel(call_tools=[], custom_output_text="Simulación")):
        primero = agente.run_sync("Me interesa una bicicleta roja.", deps=ana)
        segundo = agente.run_sync(
            "¿De qué objeto hablamos?", deps=ana,
            message_history=primero.all_messages(),
        )
        sin_historial = agente.run_sync("¿De qué objeto hablamos?", deps=ana)
        ana_detallada = Cliente(
            cliente_id="ana", nombre="Ana", presupuesto_mxn=100,
            estilo="detallado",
        )
        actualizado = agente.run_sync(
            "Responde con mi estilo actual.", deps=ana_detallada,
            message_history=primero.all_messages(),
        )

    assert any(
        isinstance(p, UserPromptPart) and p.content == "Me interesa una bicicleta roja."
        for p in partes(segundo)
    )
    assert not any(
        isinstance(p, UserPromptPart) and p.content == "Me interesa una bicicleta roja."
        for p in partes(sin_historial)
    )
    assert any(
        isinstance(p, SystemPromptPart) and "Estilo de respuesta: detallado" in p.content
        for p in partes(actualizado)
    )
    try:
        Cliente(cliente_id="invalido", nombre="Prueba", presupuesto_mxn=-1)
    except ValidationError:
        pass
    else:
        raise AssertionError("Se aceptó un presupuesto negativo.")
    print("Pruebas superadas: perfiles, herramienta, historial y prompt dinámico.")
finally:
    models.ALLOW_MODEL_REQUESTS = permitidas_antes


## 6. Configurar OpenRouter (activar USAR_OPENROUTER para la prueba real)


In [ ]:
USAR_OPENROUTER = False
MODELO = "openrouter:openai/gpt-4o-mini"

if USAR_OPENROUTER:
    import os
    from google.colab import userdata

    try:
        clave = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        raise RuntimeError(
            "No se pudo leer el secreto. Revisa su nombre y el permiso en Colab."
        ) from None
    if not clave or not clave.strip():
        raise RuntimeError("El secreto OPENROUTER_API_KEY está vacío.")
    os.environ["OPENROUTER_API_KEY"] = clave.strip()
    del clave
else:
    print("Modo sin llamadas reales. Las pruebas anteriores no necesitan clave.")


## 7. Comparar con historial, sin historial y otro cliente


In [ ]:
if USAR_OPENROUTER:
    turno_1 = agente.run_sync(
        "Me interesa una bicicleta roja. Recuérdala durante esta conversación.",
        deps=ana, model=MODELO,
    )
    print("Primer turno:", turno_1.output)

    pregunta = "¿Qué objeto me interesa y a cuántas bananas equivale mi presupuesto?"
    turno_2 = agente.run_sync(
        pregunta, deps=ana, model=MODELO,
        message_history=turno_1.all_messages(),
    )
    print("Con historial:", turno_2.output)

    sin_memoria = agente.run_sync(pregunta, deps=ana, model=MODELO)
    print("Sin historial:", sin_memoria.output)

    otro_cliente = agente.run_sync(
        "Consulta mi presupuesto y su equivalencia en bananas.",
        deps=bruno, model=MODELO,
    )
    print("Otro cliente:", otro_cliente.output)


## 8. Auditar las llamadas y los mensajes


In [ ]:
if USAR_OPENROUTER:
    from pydantic_ai.messages import ToolCallPart

    for mensaje in turno_2.new_messages():
        for parte in mensaje.parts:
            if isinstance(parte, (ToolCallPart, ToolReturnPart)):
                print(parte)
    print("Mensajes acumulados:", len(turno_2.all_messages()))
    print("Mensajes nuevos:", len(turno_2.new_messages()))


## 9. Solución del mini-reto: Carla, 175 MXN y paraguas amarillo


In [ ]:
carla = Cliente(cliente_id="carla", nombre="Carla", presupuesto_mxn=175)
permitidas_antes = models.ALLOW_MODEL_REQUESTS
models.ALLOW_MODEL_REQUESTS = False
try:
    with agente.override(model=TestModel()):
        consulta_carla = agente.run_sync("Consulta mi presupuesto.", deps=carla)
    assert any(
        isinstance(p, ToolReturnPart)
        and p.tool_name == "resumen_presupuesto"
        and p.content["bananas_enteras"] == 35
        for p in partes(consulta_carla)
    )
    with agente.override(model=TestModel(call_tools=[], custom_output_text="Simulación")):
        inicio_carla = agente.run_sync("Me interesa un paraguas amarillo.", deps=carla)
        siguiente_carla = agente.run_sync(
            "¿De qué objeto hablamos?", deps=carla,
            message_history=inicio_carla.all_messages(),
        )
    assert any(
        isinstance(p, UserPromptPart) and p.content == "Me interesa un paraguas amarillo."
        for p in partes(siguiente_carla)
    )
    print("Mini-reto superado: 35 bananas y mensaje previo transmitido.")
finally:
    models.ALLOW_MODEL_REQUESTS = permitidas_antes
